# Notebook 4: Graph Engine Validation
Goal: Validate NetworkX graph construction, Dijkstra routing, and scenario simulation.

In [ ]:
import sys
import os
import time
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = '../data/processed/'
print('Environment ready.')

## 1. Build / Load Graph Cache

In [ ]:
from src.graph.builder import build_all_graphs, load_graphs_cache

# load_graphs_cache() loads pre-built graphs from data/processed/graphs_cache/
# If cache does not exist, call build_all_graphs() to create it.
try:
    graphs = load_graphs_cache()
    print('Loaded graphs from cache.')
except FileNotFoundError:
    print('Cache not found — building all graphs...')
    graphs = build_all_graphs(save=True)
    print('Done building graphs.')

print(f'\nNumber of (year, product_code) graph snapshots: {len(graphs)}')

# Report a sample of graph sizes
print('\nSample graph sizes:')
for i, (key, G) in enumerate(graphs.items()):
    print(f'  {key}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    if i >= 4:
        print('  ...')
        break

## 2. Toy Graph Unit Tests

In [ ]:
# Build a simple 3-node directed graph A -> B -> C
# Edge A->C also exists but with higher weight to force A->B->C as Dijkstra solution

G_toy = nx.DiGraph()
G_toy.add_edge('A', 'B', weight=1.0, lead_time=5)
G_toy.add_edge('B', 'C', weight=1.0, lead_time=5)
G_toy.add_edge('A', 'C', weight=10.0, lead_time=20)  # more expensive direct route

# Run Dijkstra
shortest_path = nx.dijkstra_path(G_toy, 'A', 'C', weight='weight')
shortest_cost = nx.dijkstra_path_length(G_toy, 'A', 'C', weight='weight')

print(f'Dijkstra path A->C: {shortest_path}')
print(f'Dijkstra cost:      {shortest_cost}')

# Assertions
assert shortest_path == ['A', 'B', 'C'], f'Expected [A, B, C], got {shortest_path}'
assert abs(shortest_cost - 2.0) < 1e-9, f'Expected cost 2.0, got {shortest_cost}'
print('\nToy graph unit test: PASS')

# Test no-path case
G_toy2 = nx.DiGraph()
G_toy2.add_edge('A', 'B', weight=1.0)
try:
    nx.dijkstra_path(G_toy2, 'A', 'C', weight='weight')
    print('No-path exception test: FAIL (should have raised)')
except nx.NetworkXNoPath:
    print('No-path exception test: PASS')
except nx.NodeNotFound:
    print('No-path exception test: PASS (node not found)')

## 3. Baseline Routing Demo

In [ ]:
from src.graph.routing import find_k_routes

G = graphs.get((2021, 8517))
if G is None:
    # Fallback: use any available graph
    key = list(graphs.keys())[0]
    G = graphs[key]
    print(f'Graph (2021, 8517) not found. Using {key} instead.')
else:
    print('Loaded graph: (2021, HS 8517 — Telephones/Communication Equipment)')

print(f'Graph has {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.')

# Find top-3 routes from China to Germany
ORIGIN = 'China'
DEST   = 'Germany'
routes = find_k_routes(G, ORIGIN, DEST, k=3)

print(f'\nTop-3 routes: {ORIGIN} → {DEST} (Year 2021, HS 8517)\n')
for i, route in enumerate(routes, 1):
    path        = route.get('path', [])
    cost        = route.get('cost', float('nan'))
    lead_time   = route.get('lead_time_days', float('nan'))
    resilience  = route.get('resilience_score', float('nan'))
    print(f'  Route {i}:')
    print(f'    Path:             {" → ".join(path)}')
    print(f'    Cost (USD/TEU):   {cost:,.0f}')
    print(f'    Lead time (days): {lead_time:.1f}')
    print(f'    Resilience score: {resilience:.1f}')
    print()

## 4. Chokepoint Scenario: Suez Canal

In [ ]:
from src.graph.routing import apply_scenario, find_k_routes

# Baseline routes
baseline_routes = find_k_routes(G, 'China', 'Germany', k=3)

# Suez Canal scenario: block the Egypt node (proxy for canal closure)
suez_scenario = {
    'type': 'chokepoint',
    'blocked_nodes': ['Egypt'],          # remove Egypt from routing
    'cost_multiplier_edges': {},         # no additional cost multipliers
    'description': 'Suez Canal Closure'
}

G_suez   = apply_scenario(G, suez_scenario)
suez_routes = find_k_routes(G_suez, 'China', 'Germany', k=3)

print('=== Scenario: Suez Canal Closure ===')
print(f'Baseline top route:  {" → ".join(baseline_routes[0]["path"])} | cost: {baseline_routes[0]["cost"]:,.0f}')

if suez_routes:
    print(f'Rerouted top route:  {" → ".join(suez_routes[0]["path"])} | cost: {suez_routes[0]["cost"]:,.0f}')
    cost_prem = (suez_routes[0]['cost'] - baseline_routes[0]['cost']) / baseline_routes[0]['cost'] * 100
    print(f'\nCost premium due to Suez closure: +{cost_prem:.1f}%')
    
    # Visualise cost comparison
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(['Baseline', 'Suez Closed'],
           [baseline_routes[0]['cost'], suez_routes[0]['cost']],
           color=['steelblue', 'coral'], edgecolor='white')
    ax.set_ylabel('Top-Route Cost (USD/TEU)')
    ax.set_title('China → Germany: Suez Canal Scenario')
    for i, val in enumerate([baseline_routes[0]['cost'], suez_routes[0]['cost']]):
        ax.text(i, val + 20, f'{val:,.0f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('No alternative route found after Suez closure (fully blocked corridor).')

## 5. Tariff Scenario: US-China Trade War

In [ ]:
# Baseline routes China -> USA
baseline_us = find_k_routes(G, 'China', 'United States', k=3)

# Tariff scenario: 25% cost increase on all edges entering or leaving USA or China
tariff_scenario = {
    'type': 'tariff',
    'blocked_nodes': [],
    'cost_multiplier_edges': {
        'origin': 'China',
        'dest':   'United States',
        'multiplier': 1.25
    },
    'description': 'US-China 25% tariff'
}

G_tariff   = apply_scenario(G, tariff_scenario)
tariff_routes = find_k_routes(G_tariff, 'China', 'United States', k=3)

print('=== Scenario: US-China 25% Mutual Tariff ===')
print('Before tariff:')
for i, r in enumerate(baseline_us[:3], 1):
    print(f'  Route {i}: {" → ".join(r["path"])} | cost: {r["cost"]:,.0f}')

print('\nAfter 25% tariff:')
for i, r in enumerate(tariff_routes[:3], 1):
    print(f'  Route {i}: {" → ".join(r["path"])} | cost: {r["cost"]:,.0f}')

if baseline_us and tariff_routes:
    delta = (tariff_routes[0]['cost'] - baseline_us[0]['cost']) / baseline_us[0]['cost'] * 100
    print(f'\nTop-route cost change: {delta:+.1f}%')

    # Check if tariff causes route diversion (different transshipment ports)
    baseline_path = set(baseline_us[0]['path'])
    tariff_path   = set(tariff_routes[0]['path'])
    new_nodes     = tariff_path - baseline_path
    dropped_nodes = baseline_path - tariff_path
    if new_nodes or dropped_nodes:
        print(f'Route diversion detected!')
        print(f'  New nodes in rerouted path:     {new_nodes}')
        print(f'  Dropped nodes from baseline:    {dropped_nodes}')
    else:
        print('No route diversion — same path, higher cost due to tariff.')

## 6. Network Analysis

In [ ]:
# Betweenness centrality for the 2021 HS-8517 graph
print('Computing betweenness centrality (may take ~30 seconds for large graphs)...')
t0 = time.time()

# Use approximate betweenness for speed on large graphs (k=100 samples)
n_nodes = G.number_of_nodes()
k_samples = min(100, n_nodes)
bc = nx.betweenness_centrality(G, weight='weight', normalized=True, k=k_samples)

print(f'Done in {time.time() - t0:.1f}s.  Nodes evaluated: {n_nodes}')

bc_series = pd.Series(bc).sort_values(ascending=False)
top10 = bc_series.head(10)

print('\nTop-10 most central nodes (betweenness centrality):')
print(top10.to_string())

fig, ax = plt.subplots(figsize=(9, 4))
top10.sort_values().plot.barh(ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Betweenness Centrality')
ax.set_title('Top-10 Network Chokepoints (Betweenness Centrality)')
plt.tight_layout()
plt.show()

# Comment: known chokepoints should appear at the top
EXPECTED_CHOKEPOINTS = {'Singapore', 'Malaysia', 'Egypt', 'Panama', 'Netherlands', 'Sri Lanka'}
top10_set = set(top10.index)
found = top10_set & EXPECTED_CHOKEPOINTS
print(f'\nKnown chokepoint countries in top-10: {found}')
if found:
    print('Network structure validation: PASS — chokepoint nations are highly central.')
else:
    print('NOTE: No expected chokepoints found in top-10 — check graph construction.')

## 7. Performance Benchmark

In [ ]:
# Time 100 Dijkstra queries on the largest available graph
largest_key = max(graphs.keys(), key=lambda k: graphs[k].number_of_edges())
G_large = graphs[largest_key]
print(f'Benchmarking on graph {largest_key}: {G_large.number_of_nodes()} nodes, {G_large.number_of_edges()} edges')

nodes = list(G_large.nodes())
np.random.seed(42)
src_nodes = np.random.choice(nodes, size=100, replace=True)
tgt_nodes = np.random.choice(nodes, size=100, replace=True)

latencies = []
n_success = 0

for src, tgt in zip(src_nodes, tgt_nodes):
    if src == tgt:
        continue
    t_start = time.perf_counter()
    try:
        path = nx.dijkstra_path(G_large, src, tgt, weight='weight')
        n_success += 1
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        pass
    t_end = time.perf_counter()
    latencies.append(t_end - t_start)

mean_latency = np.mean(latencies)
p95_latency  = np.percentile(latencies, 95)

print(f'\nBenchmark results (100 queries):')
print(f'  Successful routes found: {n_success}/100')
print(f'  Mean latency:            {mean_latency*1000:.2f} ms')
print(f'  P95 latency:             {p95_latency*1000:.2f} ms')

THRESHOLD_SEC = 0.1  # 100ms SLA
if mean_latency < THRESHOLD_SEC:
    print(f'\nPerformance test: PASS (mean {mean_latency*1000:.1f}ms < {THRESHOLD_SEC*1000:.0f}ms threshold)')
else:
    print(f'\nPerformance test: FAIL (mean {mean_latency*1000:.1f}ms > {THRESHOLD_SEC*1000:.0f}ms threshold)')

assert mean_latency < THRESHOLD_SEC, f'Mean Dijkstra latency {mean_latency:.4f}s exceeds 0.1s SLA'

# Latency distribution
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist([l * 1000 for l in latencies], bins=30, color='steelblue', edgecolor='white')
ax.axvline(mean_latency * 1000, color='red', linestyle='--', label=f'Mean: {mean_latency*1000:.1f} ms')
ax.set_xlabel('Query Latency (ms)')
ax.set_ylabel('Count')
ax.set_title('Dijkstra Query Latency Distribution (n=100)')
ax.legend()
plt.tight_layout()
plt.show()